<a href="https://colab.research.google.com/github/SiyumiJayawardhane/sinhala-wsd/blob/sinhala_textbook_extraction/textbook_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Setup

*   Installs pymupdf (to read the PDF’s text) and pandukabhaya (to convert FM-Abhaya to Unicode), then connects Drive.

In [17]:
!pip install -q pymupdf pandukabhaya

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 41.0 MB/s eta 0:00:00


In [21]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, random, subprocess, unicodedata, datetime
from collections import Counter
from pandukabhaya import Converter

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


2. Load the book

In [23]:
BASE    = "/content/drive/MyDrive/Research-WSD-New"
PDF_DIR = f"{BASE}/textbooks"
OUT_DIR = f"{BASE}/textbooks/Ext_Final"

BOOKS = [
    {
        "book_id":     "sinhala_lit_g10_11",
        "pdf":         "sinhala lit g-10-11.pdf",
        "title":       "සිංහල සාහිත්‍ය සංග්‍රහය (Grade 10–11)",
        "publisher":   "Educational Publications Department",
        "grade":       "10-11",
        "year":        "",
        "url":         "",
        "page_offset": 10,
    },
]
for b in BOOKS:
    b["pdf_path"] = os.path.join(PDF_DIR, b["pdf"])
    b["out"] = os.path.join(OUT_DIR, b["book_id"]); os.makedirs(b["out"], exist_ok=True)
    print(("OK      " if os.path.exists(b["pdf_path"]) else "MISSING ") + b["pdf_path"])

OK      /content/drive/MyDrive/Research-WSD-New/textbooks/sinhala lit g-10-11.pdf


In [24]:
# Why not pdftotext: (1) it reads two-column poems column by column, so a verse line loses its
# rhyme word; (2) it can't see fonts, so English words (Times font) were "converted" into fake Sinhala.
# PyMuPDF gives each text span with its font and x/y position, so we can fix both.
import pymupdf
COL_GAP, WORD_GAP, LINE_TOL = 12.0, 1.5, 3.0

def read_spans(path):
    """List of pages; each page = list of spans {text, font, bbox}."""
    with pymupdf.open(path) as doc:
        return [[s for b in p.get_text("dict")["blocks"] for l in b.get("lines", []) for s in l["spans"]
                 if s["text"].strip()] for p in doc]

def detect_encoding(pages):
    """'fm' if most text is in FM (legacy) fonts, 'unicode' if it is already Sinhala Unicode,
    'none' if there is (almost) no text layer -> scanned -> needs Document AI OCR."""
    chars = Counter()
    for p in pages:
        for s in p:
            chars["fm" if s["font"].startswith("FM") else
                  "unicode" if SINHALA.search(s["text"]) else "other"] += len(s["text"])
    total = sum(chars.values())
    if total < 500:
        return "none", chars
    return ("fm" if chars["fm"] > chars["unicode"] else "unicode"), chars

SINHALA = re.compile(r"[\u0D80-\u0DFF]")
LATIN   = re.compile(r"[A-Za-z]")
for b in BOOKS:
    b["spans"] = read_spans(b["pdf_path"])
    b["encoding"], chars = detect_encoding(b["spans"])
    fonts = Counter(s["font"] for p in b["spans"] for s in p)
    print(f'{b["book_id"]}: {len(b["spans"])} pages -> {b["encoding"]} | chars by type {dict(chars)}')
    print("   fonts:", fonts.most_common(8))

sinhala_lit_g10_11: 140 pages -> fm | chars by type {'fm': 224027, 'other': 1912}
   fonts: [('FMAbhayax', 6420), ('TimesNewRomanPSMT', 1259), ('FMAbabldBold', 446), ('MinionPro-Regular', 144), ('FMEmaneex', 70), ('FMSamanthaBoldx', 16), ('TimesNewRomB', 4), ('TimesNewRomanPS-BoldMT', 3)]


3. Read the text and check its encoding

In [25]:
SINHALA = re.compile(r"[\u0D80-\u0DFF]")
LATIN   = re.compile(r"[A-Za-z]")

def pdf_pages(path):
    """pdftotext output split into pages (form feed = page break). Index 0 = PDF page 1."""
    raw = subprocess.run(["pdftotext", path, "-"], capture_output=True, text=True, check=True).stdout
    pages = raw.split("\f")
    return pages[:-1] if pages and not pages[-1].strip() else pages

def detect_encoding(pages):
    """'unicode' = already Sinhala Unicode; 'fm' = legacy font (Latin letters stand for Sinhala glyphs);
    'none' = no text layer (scanned) -> needs OCR (Document AI) instead."""
    text = "".join(pages)
    s, l = len(SINHALA.findall(text)), len(LATIN.findall(text))
    if s + l < 500:
        return "none", s, l
    return ("unicode" if s > l else "fm"), s, l

for b in BOOKS:
    b["raw_pages"] = pdf_pages(b["pdf_path"])
    b["encoding"], s, l = detect_encoding(b["raw_pages"])
    print(f'{b["book_id"]}: {len(b["raw_pages"])} pages | Sinhala chars {s:,} | Latin chars {l:,} -> {b["encoding"]}')
    print("   sample raw text:", re.sub(r"\s+", " ", b["raw_pages"][min(15, len(b["raw_pages"])-1)])[:150])

sinhala_lit_g10_11: 140 pages | Sinhala chars 0 | Latin chars 145,027 -> fm
   sample raw text: ye<,S .sh uyd id.rh fia fldam kue;s jd;fhka ye<,S .sh wgf<dia wlafI!ys‚hla fiakdfjka hqla; je kslauqKdy' flajÜgfhda o ;ukaf.a nia rÊcqrejka is;g kxjd 


4. Convert FM-Abhaya to Unicode

In [26]:
conv = Converter("fm_abhaya")
NOISE = re.compile(r"[\u00AD\u200B\uFEFF]")

# Glyphs the fm_abhaya table leaves unconverted in this book (checked on real words):
# `ඵ -> ළු (නෙළුම්, පුළුවන්), `ද -> ඳ (අඳින්නේ), `ඩ -> ඬ (පඬිතුමා), ¥ -> දූ (දූතයා, දූවිලි),
# i after a Sinhala letter -> ද්‍ය (විද්‍යාලය, පද්‍ය).
POST_FIXES = [(re.compile(r"`ඵ"), "ළු"), (re.compile(r"`ද"), "ඳ"), (re.compile(r"`ඩ"), "ඬ"),
              (re.compile(r"¥"), "දූ"), (re.compile(r"(?<=[\u0D80-\u0DFF])i"), "ද්\u200Dය")]

def fm_to_unicode(t):
    # “ and ‘ are used as quote marks in this book, but the table maps “ -> ර්‍ණ and ' ‘' -> ි.
    # Protect them BEFORE converting so real letters are untouched.
    t = t.replace("“", "\u0001").replace("‘", "\u0002")
    t = conv.convert(t).replace("\u0001", "“").replace("\u0002", "‘")
    for pat, rep in POST_FIXES:
        t = pat.sub(rep, t)
    return t

def segment_text(spans, encoding):
    """Text of spans that sit close together on one line. FM-font runs are converted together;
    other fonts (Times, Minion = English, ISBN) are kept exactly as they are."""
    out, run, run_fm, prev_x1 = [], "", None, None
    for s in spans:
        fm = encoding == "fm" and s["font"].startswith("FM")
        txt = s["text"]
        if prev_x1 is not None and s["bbox"][0] - prev_x1 > WORD_GAP and not txt.startswith(" "):
            txt = " " + txt
        if run_fm is None or fm == run_fm:
            run += txt; run_fm = fm
        else:
            out.append(fm_to_unicode(run) if run_fm else run); run, run_fm = txt, fm
        prev_x1 = s["bbox"][2]
    if run:
        out.append(fm_to_unicode(run) if run_fm else run)
    return re.sub(r"\s+", " ", "".join(out)).strip()

DEP_SIGN = re.compile(r"^[\u0DCA-\u0DDF\u0D82\u0D83]")
NUM_ONLY = re.compile(r"^\(?\d{1,3}[.)]?$")

GLOSS_ROW = re.compile(r"^\S[^.?!]{0,60}\s-\s")
is_short = lambda t: all(len(w) <= 2 for w in t.split())

def merge_segments(segs):
    """Pieces of one printed line (left to right, already without a second column) -> (lines, new_unit).
    - rhyme word / syllables printed apart on the right of a verse line -> joined to the line
      (ගොදු + රු = ගොදුරු, පඳුරු + ත ළා = පඳුරු තළා)
    - glossary 'word   - meaning' -> one row; two glossary entries on one printed line -> two rows
    - label 'අන්වය :   text' -> one line
    A stanza / question number at the start ("28.") or a glossary row starts a new unit."""
    new_unit = False
    if len(segs) > 1 and NUM_ONLY.match(segs[0]):
        segs, new_unit = segs[1:], True
    out, cur = [], segs[0]
    for i, nxt in enumerate(segs[1:], start=1):
        next_is_meaning = i + 1 < len(segs) and segs[i + 1].startswith("-")
        if " - " in cur and next_is_meaning:
            out.append(cur); cur = nxt
        elif DEP_SIGN.match(nxt) or (is_short(nxt) and len(cur.split()[-1]) <= 1):
            cur += nxt.replace(" ", "")
        elif is_short(nxt):
            cur += " " + nxt.replace(" ", "")
        else:
            cur += " " + nxt
    out.append(cur)
    return out, new_unit or bool(GLOSS_ROW.match(out[0]))

def rebuild_page(spans, encoding):
    """Group spans into printed lines (same baseline), split lines at big gaps, merge, and return the
    page text: left column first, then the right column (only where a page really has two)."""
    spans = sorted(spans, key=lambda s: (round(s["bbox"][3]), s["bbox"][0]))
    kept = []                                    # drop "fake bold" duplicates: same text printed twice
    for s in spans:                              # a few points apart (shadow effect)
        if not any(k["text"].replace(" ", "") == s["text"].replace(" ", "") and abs(k["bbox"][0] - s["bbox"][0]) < 5
                   and abs(k["bbox"][3] - s["bbox"][3]) < 4 for k in kept[-6:]):
            kept.append(s)
    spans, rows = kept, []
    for s in spans:
        if rows and abs(rows[-1][0] - s["bbox"][3]) <= LINE_TOL:
            rows[-1][1].append(s)
        else:
            rows.append([s["bbox"][3], [s]])
    # pass 1: cut every printed line into pieces at big gaps, remembering where each piece starts (x)
    lines = []
    for _, ss in rows:
        ss.sort(key=lambda s: s["bbox"][0])
        groups, cur = [], [ss[0]]
        for s in ss[1:]:
            if s["bbox"][0] - cur[-1]["bbox"][2] > COL_GAP:
                groups.append(cur); cur = [s]
            else:
                cur.append(s)
        groups.append(cur)
        segs = [(segment_text(g, encoding), g[0]["bbox"][0]) for g in groups]
        lines.append([(t, x) for t, x in segs if t])
    # pass 2: find a real two-column block (e.g. a two-column song). A "right-column start" is a piece
    # that begins right of 45% of the page width and is several real words — not a rhyme word, not
    # spaced rhyme syllables, not a glossary meaning. If a page has 4+ such lines, every piece that
    # starts at or right of that column, between the first and last such line, goes to column 2.
    width = max((s["bbox"][2] for s in spans), default=0)
    def col2_start(t, x):
        return x > 0.45 * width and " " in t and not is_short(t) and not t.startswith("-")
    hits = [i for i, segs in enumerate(lines)
            if any(col2_start(t, x) for t, x in segs) and not any(t.startswith("-") for t, _ in segs)]
    if len(hits) >= 4:
        boundary = min(x for i in hits for t, x in lines[i] if col2_start(t, x)) - 10
        lo, hi = hits[0], hits[-1]
    else:
        boundary, lo, hi = width + 1, -1, -1
    streams = {}
    for i, segs in enumerate(lines):
        if not segs:
            continue
        in_block = lo <= i <= hi
        to_col2 = lambda t, x: in_block and x >= boundary and not is_short(t)   # rhyme syllables stay
        left = [t for t, x in segs if not to_col2(t, x)]
        rcol = " ".join(t for t, x in segs if to_col2(t, x))
        rows, new_unit = merge_segments(left) if left else ([], False)
        targets = [(0, t) for t in rows] + ([(1, rcol)] if rcol else [])
        for col, text in targets:
            st = streams.setdefault(col, [])
            if new_unit or (col == 0 and GLOSS_ROW.match(text)):
                st.append("")                      # blank line = hard boundary before this line
            st.append(text)
            if col == 0 and GLOSS_ROW.match(text):
                st.append("")
    page = "\n\n".join("\n".join(streams[k]) for k in sorted(streams))
    return NOISE.sub("", unicodedata.normalize("NFC", page))

for b in BOOKS:
    if b["encoding"] == "none":
        print(f'{b["book_id"]}: no text layer — skip here, use Document AI OCR'); continue
    b["pages"] = [rebuild_page(p, b["encoding"]) for p in b["spans"]]
    json.dump(b["pages"], open(f'{b["out"]}/{b["book_id"]}_pages_unicode.json', "w", encoding="utf-8"), ensure_ascii=False)
    print(f'{b["book_id"]}: {len(b["pages"])} pages converted')
    print("   sample:", re.sub(r"\s+", " ", b["pages"][min(40, len(b["pages"])-1)])[:200])

sinhala_lit_g10_11: 140 pages converted
   sample: (2) වනපෙතිනැවිදින් : වනපෙතින් + ඇවිදින් = වනපෙතිනැවිදින් (3) සුමුවෝ : සමූහ ව (4) නදළ මුවෝ : නද + අළ මුවෝ = නදළ මුවෝ අළ යනු අර (ඇරීමේ) ධාතුවේ අතීත කෘදන්තයි. නාදයට විස්මිත ව කට ඇර ගෙන සිටියා වූ අළ මුව ය


5. Check the conversion page by page

In [27]:
# After a good conversion almost no Latin letters remain. Leftover Latin = glyphs the converter
# did not map (or real English text on that page) have to look at those pages before using them.
LATIN_LEFT_MAX = 0.02          # flag a page if > 2% of its letters are still Latin

rows = []
for b in BOOKS:
    if "pages" not in b: continue
    leftovers = Counter()
    for i, p in enumerate(b["pages"]):
        s, l = len(SINHALA.findall(p)), len(LATIN.findall(p))
        leftovers.update(ch for ch in p if not (SINHALA.match(ch) or ch.isspace() or ch.isdigit()
                                               or ch in ".,;:!?()[]-–—'\"“”‘’/\u200C\u200D"))
        rows.append({"book": b["book_id"], "pdf_page": i + 1, "sinhala": s, "latin": l,
                     "flag": (s + l) > 0 and l / (s + l) > LATIN_LEFT_MAX})
    flagged = [r["pdf_page"] for r in rows if r["book"] == b["book_id"] and r["flag"]]
    print(f'{b["book_id"]}: {len(flagged)} flagged pages {flagged[:20]}')
    print("   most common leftover characters:", leftovers.most_common(15))

import pandas as pd
pd.DataFrame(rows).to_csv(f"{OUT_DIR}/conversion_page_check.csv", index=False, encoding="utf-8")

sinhala_lit_g10_11: 6 flagged pages [2, 4, 5, 8, 53, 54]
   most common leftover characters: [('i', 35), ('+', 30), ('e', 28), ('n', 24), ('a', 23), ('l', 17), ('t', 17), ('h', 12), ('=', 12), ('u', 11), ('o', 11), ('r', 11), ('g', 10), ('s', 8), ('d', 7)]


6. Split into sentences


*   Page-number lines are dropped.
*   Prose is split at . ? !.
*   Long poem chunks are cut into 2-line units, because poems have no full stops.

In [28]:
#  - lines that are only page numbers (digits / roman numerals) are dropped
#  - the book's study notes start with labels අන්වය : / අර්ථය : / විස්තර : -> each label starts a new unit,
#    so a verse stanza is never glued to its explanation
#  - prose is split after . ? !
#  - verse has no full stops: a chunk > 280 chars with >= 3 lines is cut into units of
#    VERSE_LINES lines
#  - units with fewer than 4 Sinhala letters are dropped (numbering "03.", initials "එම්.")
VERSE_LINES = 2
PAGE_NUM = re.compile(r"[\divxlcIVXLC\s]+")
SPLIT    = re.compile(r"(?<=[.?!])\s+")
LABEL    = re.compile(r"\s*(?=(?:අන්වය|අර්ථය|විස්තර)\s*\*?\s*:)")

def split_units(pages, offset):
    units = []
    for pi, p in enumerate(pages):
        lines = [re.sub(r"[ \t]+", " ", l).strip() for l in p.split("\n")]
        lines = [l for l in lines if not PAGE_NUM.fullmatch(l)]
        text = LABEL.sub("\n\n", "\n".join(lines))
        text = re.sub(r"\n{3,}", "\n\n", text)
        for block in text.split("\n\n"):
            for s in SPLIT.split(block):
                s = s.strip()
                if not s: continue
                if len(s) > 280 and s.count("\n") >= 2:
                    ls = s.split("\n")
                    chunks = [" ".join(ls[i:i + VERSE_LINES]) for i in range(0, len(ls), VERSE_LINES)]
                else:
                    chunks = [s.replace("\n", " ")]
                for c in chunks:
                    c = re.sub(r"\s+", " ", c).strip()
                    if len(SINHALA.findall(c)) >= 4:
                        bp = pi + 1 - offset
                        units.append({"sid": len(units), "pdf_page": pi + 1,
                                      "book_page": bp if bp >= 1 else "", "text": c})
    return units

for b in BOOKS:
    if "pages" not in b: continue
    b["units"] = split_units(b["pages"], b["page_offset"])
    with open(f'{b["out"]}/{b["book_id"]}_sentences.jsonl', "w", encoding="utf-8") as f:
        for u in b["units"]:
            f.write(json.dumps({"book_id": b["book_id"], **u}, ensure_ascii=False) + "\n")
    with open(f'{b["out"]}/{b["book_id"]}_corpus.txt', "w", encoding="utf-8") as f:   # one unit per line
        f.write("\n".join(u["text"] for u in b["units"]))
    print(f'{b["book_id"]}: {len(b["units"]):,} sentence units -> {b["book_id"]}_corpus.txt')

sinhala_lit_g10_11: 3,190 sentence units -> sinhala_lit_g10_11_corpus.txt


7. Spot-check sheet

In [29]:
SPOT_N, SEED = 100, 42
for b in BOOKS:
    if "units" not in b: continue
    sample = random.Random(SEED).sample(b["units"], min(SPOT_N, len(b["units"])))
    pd.DataFrame([{"sid": u["sid"], "pdf_page": u["pdf_page"], "book_page": u["book_page"],
                   "text": u["text"], "ok (y/n)": "", "error type": "", "note": ""}
                  for u in sorted(sample, key=lambda u: u["sid"])]
                 ).to_excel(f'{b["out"]}/{b["book_id"]}_spot_check.xlsx', index=False)
    print(f'{b["book_id"]}: {len(sample)} units -> {b["book_id"]}_spot_check.xlsx')

sinhala_lit_g10_11: 100 units -> sinhala_lit_g10_11_spot_check.xlsx


In [30]:
import importlib.metadata as md
for b in BOOKS:
    if "units" not in b: continue
    meta = {k: b[k] for k in ["book_id", "pdf", "title", "publisher", "grade", "year", "url", "page_offset", "encoding"]}
    meta.update({
        "method": "PyMuPDF text spans (font + position); FM-font spans converted with pandukabhaya (fm_abhaya), "
                  "English (Times/Minion) spans kept; lines rebuilt by position (two-column poems, rhyme words); NFC"
                  if b["encoding"] == "fm" else "PyMuPDF text spans (already Unicode); lines rebuilt by position; NFC",
        "quote_fix": "FM glyphs “ and ‘ kept as quote marks before conversion" if b["encoding"] == "fm" else "",
        "pymupdf": md.version("pymupdf"),
        "pandukabhaya": md.version("pandukabhaya"),
        "pages": len(b["pages"]), "sentence_units": len(b["units"]),
        "date": datetime.date.today().isoformat(),
    })
    json.dump(meta, open(f'{b["out"]}/{b["book_id"]}_meta.json', "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(json.dumps(meta, ensure_ascii=False, indent=1))

{
 "book_id": "sinhala_lit_g10_11",
 "pdf": "sinhala lit g-10-11.pdf",
 "title": "සිංහල සාහිත්‍ය සංග්‍රහය (Grade 10–11)",
 "publisher": "Educational Publications Department",
 "grade": "10-11",
 "year": "",
 "url": "",
 "page_offset": 10,
 "encoding": "fm",
 "method": "PyMuPDF text spans (font + position); FM-font spans converted with pandukabhaya (fm_abhaya), English (Times/Minion) spans kept; lines rebuilt by position (two-column poems, rhyme words); NFC",
 "quote_fix": "FM glyphs “ and ‘ kept as quote marks before conversion",
 "pymupdf": "1.28.2",
 "pandukabhaya": "1.0.1",
 "pages": 140,
 "sentence_units": 3190,
 "date": "2026-10-14"
}
